# Hybrid KD Android Deployment Lock

Khóa package encoder/decoder, vocabulary, deployment contract và model card để
đưa vào ứng dụng Android. Notebook không benchmark; benchmark được thực hiện
sau khi tích hợp app.

## Input bắt buộc — đúng 2 output

1. Output hoàn tất `HYB_INT8_Full_Validation`;
2. output `HYB_INT8_Preflight`.

Runtime: `CPU / Accelerator None / Internet Off`.

In [1]:
import csv
import hashlib
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", "/kaggle/working/HYB_Android_Lock"
))
PACKAGE = OUT / "android_package"
PACKAGE.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

def exactly_one(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (name, matches)
    return matches[0]

quality_marker_path = exactly_one(
    "HYB_INT8_FULL_VALIDATION_COMPLETE.json"
)
quality_marker = read_json(quality_marker_path)
assert quality_marker["status"] == "complete"
assert int(quality_marker["seed"]) == 2026
assert quality_marker["completed_images"] == 5000
assert quality_marker["android_benchmark_authorized"] is True
assert quality_marker["new_test_opening"] is False

authorization_path = exactly_one(
    "HYB_ANDROID_BENCHMARK_AUTHORIZED.json"
)
authorization = read_json(authorization_path)
assert authorization["status"] == "authorized"
assert int(authorization["seed"]) == 2026
assert authorization["scientific_role"] == "ablation"
assert authorization["product_deployment_override"] is True

preflight_marker_path = exactly_one(
    "HYB_INT8_PREFLIGHT_COMPLETE.json"
)
preflight_root = preflight_marker_path.parent

copied = {}
for role, reference in authorization["selected_artifacts"].items():
    source = preflight_root / reference["relative_path"]
    assert source.is_file(), source
    assert source.stat().st_size == int(reference["size_bytes"])
    assert sha256(source) == reference["sha256"]
    suffix = source.suffix
    target_name = {
        "encoder": "hybrid_s1_encoder.onnx",
        "decoder": "hybrid_s1_decoder.onnx",
        "vocabulary": "word_vocabulary_minfreq5.json",
        "deployment_contract": "hybrid_deployment_contract.json",
    }[role]
    target = PACKAGE / target_name
    shutil.copy2(source, target)
    copied[role] = {
        "relative_path": target.relative_to(OUT).as_posix(),
        "size_bytes": target.stat().st_size,
        "sha256": sha256(target),
    }

model_card = {
    "schema_version": "hyb_android_model_card_v1",
    "created_at_utc": now(),
    "model_id": "mscoco_hybrid_t1_s1",
    "display_name_vi": "T1→S1 Hybrid KD",
    "display_name_en": "T1→S1 Hybrid KD",
    "method": "Hybrid Sequence and Dense-Mapped Logit Knowledge Distillation",
    "dataset": "MS COCO 2014",
    "student": "MobileNetV3-Large + Transformer decoder",
    "seed": 2026,
    "precision": authorization["selected_precision"],
    "scientific_role": "ablation",
    "product_deployment_override": True,
    "validation_metrics_x100": {
        "CIDEr-D": 110.07,
    },
    "artifacts": copied,
    "quality_gate_passed": authorization["quality_gate_passed"],
    "held_out_test_used": False,
}
model_card_path = PACKAGE / "hybrid_model_card.json"
write_json(model_card_path, model_card)
copied["model_card"] = {
    "relative_path": model_card_path.relative_to(OUT).as_posix(),
    "size_bytes": model_card_path.stat().st_size,
    "sha256": sha256(model_card_path),
}

manifest_path = OUT / "MANIFEST.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["role", "relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    for role, row in copied.items():
        writer.writerow({"role": role, **row})

lock = {
    "schema_version": "hyb_android_deployment_lock_v1",
    "created_at_utc": now(),
    "status": "locked",
    "decision": "authorize_hybrid_mobile_integration_by_user_product_override",
    "model_id": "mscoco_hybrid_t1_s1",
    "selected_seed": 2026,
    "selected_precision": authorization["selected_precision"],
    "quality_gate_passed": authorization["quality_gate_passed"],
    "package_manifest": {
        "relative_path": manifest_path.name,
        "sha256": sha256(manifest_path),
        "rows": len(copied),
    },
    "scientific_role": "ablation",
    "scientific_primary_method_unchanged": "Sequence KD T1->S1",
    "product_deployment_override": True,
    "android_integration_authorized": True,
    "physical_device_benchmark_required": True,
    "test_set_used": False,
}
write_json(OUT / "HYB_ANDROID_DEPLOYMENT_LOCK.json", lock)

print("Android deployment lock : COMPLETE")
print("Selected precision       :", authorization["selected_precision"])
print("Quality gate passed      :", authorization["quality_gate_passed"])
print("Package artifacts        :", len(copied))
print("Next                     : copy package into Android app and benchmark")

Android deployment lock : COMPLETE
Selected precision       : INT8_DYNAMIC
Quality gate passed      : True
Package artifacts        : 5
Next                     : copy package into Android app and benchmark
